# Module 5 / Class 6 LAB -- Customer Segmentation Pipeline

**Objectives:**
- Build a complete unsupervised learning pipeline end to end
- Perform EDA on real-world-style data
- Apply K-Means with proper evaluation (elbow + silhouette)
- Visualize clusters with PCA
- Profile and interpret segments

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

np.random.seed(42)
sns.set_style('whitegrid')

## 1. Load the Mall Customers Dataset

In [ ]:
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/Mall_Customers.csv"

try:
    df = pd.read_csv(url)
except Exception:
    # Fallback: generate equivalent synthetic data
    print("URL unavailable -- generating equivalent synthetic data.")
    rng = np.random.RandomState(42)
    n = 200
    df = pd.DataFrame({
        'CustomerID': range(1, n + 1),
        'Gender': rng.choice(['Male', 'Female'], n),
        'Age': rng.randint(18, 70, n),
        'Annual Income (k$)': rng.randint(15, 140, n),
        'Spending Score (1-100)': rng.randint(1, 100, n),
    })

print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()
print()
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['Age'], bins=15, edgecolor='black', color='steelblue')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')

axes[1].hist(df['Annual Income (k$)'], bins=15, edgecolor='black', color='coral')
axes[1].set_title('Annual Income Distribution')
axes[1].set_xlabel('Annual Income (k$)')

axes[2].hist(df['Spending Score (1-100)'], bins=15, edgecolor='black', color='seagreen')
axes[2].set_title('Spending Score Distribution')
axes[2].set_xlabel('Spending Score')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['Annual Income (k$)'], df['Spending Score (1-100)'],
            c='steelblue', s=30, edgecolors='black', linewidths=0.5, alpha=0.7)
plt.title('Income vs Spending Score')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.grid(True, alpha=0.3)
plt.show()

## 3. Feature Selection and Scaling

In [ ]:
# Use Income and Spending Score for clustering
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Mean after scaling: {X_scaled.mean(axis=0).round(4)}")
print(f"Std after scaling:  {X_scaled.std(axis=0).round(4)}")

## 4. K-Means -- Elbow + Silhouette

In [ ]:
K_range = range(2, 9)
inertias = []
sil_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(list(K_range), inertias, 'bo-', linewidth=2)
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].grid(True, alpha=0.3)

axes[1].plot(list(K_range), sil_scores, 'ro-', linewidth=2)
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_k = list(K_range)[np.argmax(sil_scores)]
print(f"Best k by silhouette: {best_k} (score={max(sil_scores):.4f})")

## 5. Fit Final Model

In [ ]:
# Use k=5 (typical result for this dataset -- adjust if your elbow/silhouette says otherwise)
k_final = 5
kmeans_final = KMeans(n_clusters=k_final, n_init=10, random_state=42)
df['Cluster'] = kmeans_final.fit_predict(X_scaled)

print(f"Cluster distribution:")
print(df['Cluster'].value_counts().sort_index())

## 6. PCA Visualization

In [ ]:
# With only 2 features, PCA is mainly for demonstration.
# In practice, you would use PCA when you have many features.
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total explained: {pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['Cluster'],
                      cmap='tab10', s=40, edgecolors='black', linewidths=0.5, alpha=0.8)

# Plot centroids in PCA space
centroids_pca = pca.transform(kmeans_final.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
            s=200, c='yellow', marker='X', edgecolors='black', linewidths=2, label='Centroids')

plt.title(f'Customer Segments (k={k_final})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.show()

## 7. Cluster Scatter Plot (Original Feature Space)

In [ ]:
plt.figure(figsize=(8, 6))
for cluster_id in range(k_final):
    mask = df['Cluster'] == cluster_id
    plt.scatter(df.loc[mask, 'Annual Income (k$)'],
                df.loc[mask, 'Spending Score (1-100)'],
                s=40, edgecolors='black', linewidths=0.5,
                label=f'Cluster {cluster_id}', alpha=0.8)

plt.title('Customer Segments -- Income vs Spending')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Segment Profiling

In [ ]:
profile = df.groupby('Cluster')[['Age', 'Annual Income (k$)', 'Spending Score (1-100)']].mean().round(1)
profile['Count'] = df.groupby('Cluster')['CustomerID'].count()
profile

In [ ]:
# Visualize segment profiles
profile_plot = profile[['Annual Income (k$)', 'Spending Score (1-100)']]
profile_plot.plot(kind='bar', figsize=(10, 5), edgecolor='black')
plt.title('Segment Profiles -- Mean Income and Spending Score')
plt.xlabel('Cluster')
plt.ylabel('Value')
plt.xticks(rotation=0)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---

## TODO: Student Work

### Task 1 -- Name Each Segment

Based on the profiling table and plots above, give each cluster a descriptive business name.
For example: "High Income, Low Spenders" or "Budget-Conscious Shoppers".

# TODO: Name your segments

| Cluster | Name | Description |
|---------|------|-------------|
| 0 | ... | ... |
| 1 | ... | ... |
| 2 | ... | ... |
| 3 | ... | ... |
| 4 | ... | ... |

### Task 2 -- Business Recommendations

For each segment, write one specific marketing recommendation.
Think about: What products would you promote? What offers would you make? Which segment is the highest priority?

# TODO: Write your business recommendations

**Cluster 0 (...):**
- Recommendation: ...

**Cluster 1 (...):**
- Recommendation: ...

**Cluster 2 (...):**
- Recommendation: ...

**Cluster 3 (...):**
- Recommendation: ...

**Cluster 4 (...):**
- Recommendation: ...